In [208]:
import os
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from reportlab.lib.pagesizes import landscape, letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image, KeepTogether, PageBreak
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter

# ================== CREATE TEMP FOLDER ==================
img_path_dir = os.path.join(os.getcwd(), "temp")

if not os.path.exists(img_path_dir):
    os.makedirs(img_path_dir)
#print(os.getcwd())

# ================== DOWNLOAD ONCE ==================
def download_historical_data(tickers, start, end):
    print(f"Downloading data from {start} to {end} ...")

    end_date_plus_one = (datetime.strptime(end, '%Y-%m-%d') + timedelta(days=1)).strftime('%Y-%m-%d')
    data = yf.download(tickers, start=start, end=end_date_plus_one, progress=False, auto_adjust=True)
        
    if isinstance(data.columns, pd.MultiIndex):
        volm = data['Volume'].copy()
        data = data['Close'].copy()
    else:
        volm = pd.DataFrame()
        
    data.columns = [str(col) for col in data.columns]
    if not volm.empty:
        volm.columns = [str(col) for col in volm.columns]
    
    print(f"Download completed. Shape: {data.shape}")
    return data, volm

# ================== PRICE MONITOR TABLE ==================
def monitor_tickets(df, tickers):
    print(f"\n=== Credit Risk Monitor - {datetime.now().strftime('%Y-%m-%d %H:%M')} ===\n")
       
    data_price = []
    for ticker in tickers:
        if ticker not in df.columns:
            print(f"Warning: Ticker {ticker} not found.")
            continue
            
        df_mov =df[[ticker]].copy()
        current = df_mov[ticker].iloc[-1] 
                
        # 1-week change (~5 trading days)
        hist_1w = df_mov[ticker].tail(6) 
        week_change = ((current - hist_1w.iloc[0]) / hist_1w.iloc[0] * 100) if not hist_1w.empty else 0 
                       
        # 1-month change
        hist_1m = df_mov[ticker].tail(31) 
        month_change = ((current - hist_1m.iloc[0]) / hist_1m.iloc[0] * 100) if not hist_1m.empty else 0

        # 3-month 1 quarter change
        hist_1q = df_mov[ticker].tail(91) 
        quart_change = ((current - hist_1q.iloc[0]) / hist_1q.iloc[0] * 100) if not hist_1q.empty else 0

        # 1-year range
        hist_1y = df_mov[ticker].tail(366) 
        year_change = ((current - hist_1y.iloc[0]) / hist_1y.iloc[0] * 100) if not hist_1y.empty else 0
                    
        # 5-year range
        hist_5y = df_mov[ticker].tail(1826)
        year5_change = ((current - hist_5y.iloc[0]) / hist_5y.iloc[0] * 100) if not hist_5y.empty else 0
                    
        low_52  = df_mov[ticker].tail(366).min()
        high_52 = df_mov[ticker].tail(366).max()

        # Moving Averages                 
        df_mov['MA9'] = df_mov[ticker].rolling(window=9, min_periods=1).mean()
        df_mov['MA20'] = df_mov[ticker].rolling(window=20, min_periods=3).mean()
        df_mov['MA50'] = df_mov[ticker].rolling(window=50, min_periods=10).mean()
        df_mov['MA200'] = df_mov[ticker].rolling(window=200, min_periods=30).mean()
        df_l = df_mov.iloc[-1]
                       
        price_row = {
            'Ticker': ticker,
            'Last Close': round(current, 2),
            '  1W %  ': f"{week_change:.1f}%",
            '  1M %  ': f"{month_change:.1f}%",
            '  1Q %  ': f"{quart_change:.1f}%",
            '  1Y %  ': f"{year_change:.1f}%",
            '  5Y %  ': f"{year5_change:.1f}%",      
            '  MA9'   : round(df_l['MA9'], 2) if pd.notna(df_l['MA9']) else "N/A",
            '  MA20'  : round(df_l.get('MA20'), 2) if pd.notna(df_l.get('MA20')) else "N/A",
            '  MA50'  : round(df_l.get('MA50'), 2) if pd.notna(df_l.get('MA50')) else "N/A",
            '  MA200' : round(df_l.get('MA200'), 2) if pd.notna(df_l.get('MA200')) else "N/A",
            '52W Low-High': f"{low_52:.1f} - {high_52:.1f}",
        }
        data_price.append(price_row)
      
    if data_price:
        df_price = pd.DataFrame(data_price)
        print(df_price.to_string(index=False))
        return df_price
    else:
        print("No data retrieved for this group.")
        return None


# ================== PLOT PRICE WITH INDICATORS ==================
def plot_price_only(df_price, df_volume, tickers, periods, base_path="ma", save_path=None):
    plt.close('all')
    plt.rcdefaults()                    
    plt.rcParams.update({
        'font.size': 9.5,
        'axes.titlesize': 11,
        'axes.labelsize': 9.5,
        'xtick.labelsize': 8.5,
        'ytick.labelsize': 8.5,
        'legend.fontsize': 8.5,
        'lines.linewidth': 1.5,
        'figure.autolayout': True
    })

    df_price = df_price.copy()
    df_volume = df_volume.copy() if df_volume is not None else pd.DataFrame()
    
    for t in tickers:
        if t not in df_price.columns:
            continue
            
        df_m = df_price[[t]].copy()   
        
        # Moving Averages
        df_m['MA9']   = df_m[t].rolling(window=9,   min_periods=1).mean()
        df_m['MA20']  = df_m[t].rolling(window=20,   min_periods=3).mean()
        df_m['MA50']  = df_m[t].rolling(window=50,   min_periods=10).mean()
        df_m['MA200'] = df_m[t].rolling(window=200,   min_periods=30).mean()
                
        # MACD(12,26,9)
        ema12 = df_m[t].ewm(span=12, adjust=False).mean()
        ema26 = df_m[t].ewm(span=26, adjust=False).mean()
        df_m['MACD'] = ema12 - ema26
        df_m['Signal'] = df_m['MACD'].ewm(span=9, adjust=False).mean()
        df_m['MACD_Hist'] = df_m['MACD'] - df_m['Signal']  
        
        # RSI(14)
        delta = df_m[t].diff()
        gain = delta.where(delta > 0, 0).rolling(14).mean()
        loss = -delta.where(delta < 0, 0).rolling(14).mean()
        rs = gain / loss
        df_m['RSI'] = 100 - (100 / (1 + rs))

        has_volume = t in df_volume.columns
        df_vl = df_volume[[t]].copy() if has_volume else None
    
        for period_idx, (start_date, end_date) in enumerate(periods):
            df_t = df_m[(df_m.index >= pd.to_datetime(start_date)) & 
                        (df_m.index <= pd.to_datetime(end_date))].copy()
            if df_t.empty:
                continue   
                
            df_v = None
            if has_volume and df_vl is not None:
                df_v = df_vl[(df_vl.index >= pd.to_datetime(start_date)) & 
                             (df_vl.index <= pd.to_datetime(end_date))].copy()
    
            # latest values for legend
            latest = df_t.iloc[-1]
            latest_price = round(latest[t], 2)
            latest_ma9   = round(latest['MA9'], 2) if pd.notna(latest['MA9']) else "N/A"
            latest_ma20  = round(latest['MA20'], 2) if pd.notna(latest['MA20']) else "N/A"
            latest_ma50  = round(latest['MA50'], 2) if pd.notna(latest['MA50']) else "N/A"
            latest_ma200 = round(latest['MA200'], 2) if pd.notna(latest['MA200']) else "N/A"

            fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(14, 13), sharex=True, 
                                           gridspec_kw={'height_ratios': [5, 2, 1.6, 1]})  
    
            # Price Plot
            ax1.plot(df_t.index, df_t[t], label=f'Close Price({latest_price})', color='black', linewidth=2)
            ax1.plot(df_t.index, df_t['MA9'],   label=f'MA9 ({latest_ma9})',   color='red', linewidth=1.2)
            ax1.plot(df_t.index, df_t['MA20'],  label=f'MA20 ({latest_ma20})',  color='green',  linewidth=1.2)
            ax1.plot(df_t.index, df_t['MA50'],  label=f'MA50 ({latest_ma50})',  color='blue',    linewidth=1.2)
            ax1.plot(df_t.index, df_t['MA200'], label=f'MA200 ({latest_ma200})', color='darkviolet', linewidth=1.2)
            ax1.set_title(f"{t} - Price Trend with MAs, MACD, RSI & Volume", fontsize=13)
            ax1.set_ylabel("Price", fontsize=12)
            ax1.grid(True, alpha=0.3)
            ax1.legend(fontsize=9, loc='upper left')
            ax1.yaxis.set_major_locator(plt.MaxNLocator(12))   # ← Increase this number for more y-ticks

            
            # MACD 
            ax2.plot(df_t.index, df_t['MACD'], label='MACD(12,26)', color='black', linewidth=1.8)
            ax2.plot(df_t.index, df_t['Signal'], label='Signal(9)', color='red', linewidth=1.2)
            colors_hist = ['green' if v >= 0 else 'red' for v in df_t['MACD_Hist']]
            ax2.bar(df_t.index, df_t['MACD_Hist'], color=colors_hist, alpha=0.75, width=0.8)
            ax2.axhline(y=0, color='teal', linestyle='--', alpha=0.6)         

            macd_max = df_t['MACD'].max()
            macd_min = df_t['MACD'].min()
            max_abs = max(abs(macd_max), abs(macd_min)) 
            if max_abs>0:
                ax2.set_ylim(-max_abs, max_abs)
            else: 
                ax2.set_ylim(-0.5, 0.5)
            ax2.set_ylabel("MACD")
            ax2.grid(True, alpha=0.3)
            ax2.legend(fontsize=9, loc='upper left')

            
            # RSI(14)
            ax3.plot(df_t.index, df_t['RSI'], label='RSI(14)', color='black', linewidth=1.5)
            ax3.axhline(y=70, color='olive', linestyle='--', alpha=0.6)
            ax3.axhline(y=30, color='olive', linestyle='--', alpha=0.6)
            ax3.set_ylim(0, 100)
            ax3.set_ylabel("RSI")
            ax3.grid(True, alpha=0.3)
            ax3.legend(fontsize=9, loc='upper left')

            
            # Volume
            if df_v is not None and not df_v.empty:
                max_vol = df_v[t].max() * 1.05   
                ax4.set_ylim(0, max_vol if max_vol > 0 else 1)      # Normal case with headroom
                ax4.set_ylabel("Volume", fontsize=11)
                ax4.grid(True, alpha=0.3)
                ax4.bar(df_v.index, df_v[t], color='gray', alpha=0.8, width=0.8)
            else:
                ax4.text(0.5, 0.5, "No Volume Data", ha='center', va='center', transform=ax4.transAxes)
    
            ax4.set_xlabel("Date", fontsize=10)
            ax4.xaxis.set_major_formatter(DateFormatter('%y%m%d'))
            ax4.xaxis.set_major_locator(plt.MaxNLocator(25))   
            plt.xticks(rotation=45, fontsize=10)
            plt.yticks(fontsize=9)

            plt.subplots_adjust(left=0.08, right=0.6, top=0.36, bottom=0.1, hspace=0.01)
            
            img_path = os.path.join(img_path_dir, f"{t}_{base_path}_p{period_idx+1}.png")
            plt.savefig(img_path, dpi=200, bbox_inches='tight')
            plt.close(fig)  

            if save_path is not None:
                save_path.append((t, img_path))
    return save_path if save_path is not None else []

    
# ================== COMPARISON PLOT ==================
def plot_comparison(df, ticker_groups, periods, base_path="comparison", save_path=None):
    plt.close('all')
    plt.rcdefaults()                   
    plt.rcParams.update({
        'font.size': 9.5,
        'axes.titlesize': 11,
        'axes.labelsize': 9.5,
        'xtick.labelsize': 8.5,
        'ytick.labelsize': 8.5,
        'legend.fontsize': 8.5,
        'lines.linewidth': 1.5,
        'figure.autolayout': True
    })

    df = df.copy()
    df.index = pd.to_datetime(df.index)
    
    for i, group in enumerate(ticker_groups):       
        for period_idx, (start_date, end_date) in enumerate(periods):
            df_p = df[(df.index >= pd.to_datetime(start_date)) & 
                      (df.index <= pd.to_datetime(end_date))].copy()
            
            if df_p.empty:
                continue   

            fig = plt.figure(figsize=(14, 12))

            ax1 = plt.subplot(2, 1, 1)
            for t in group:
                if t in df_p.columns:
                    ax1.plot(df_p.index, df_p[t], label=t)
            ax1.set_title(f"Price Trend - {' & '.join(group)} ({start_date} ~ {end_date})", fontsize=13)
            ax1.set_ylabel("Price", fontsize=10)
            ax1.grid(True, alpha=0.3)
            ax1.legend(fontsize=9, loc='upper left')

            ax2 = plt.subplot(2, 1, 2)
            for t in group:
                if t in df_p.columns:
                    series = df_p[t].dropna()
                    if not series.empty:
                        scaled = (series - series.quantile(0.01)) / (series.quantile(0.99) - series.quantile(0.01)) if series.quantile(0.99)!= series.quantile(0.01) else series * 0
                        ax2.plot(series.index, scaled, label=t)
                        ax2.legend(fontsize=9, loc='upper left')
            
            ax2.set_title("Scaled Price Trend", fontsize=13)
            ax2.set_ylim(-0.2, 1.2)
            ax2.set_ylabel("Scaled (0-1)", fontsize=11)
            ax2.grid(True, alpha=0.3)

            for ax in [ax1, ax2]:
                ax.xaxis.set_major_formatter(DateFormatter('%y%m%d'))
                ax.xaxis.set_major_locator(plt.MaxNLocator(25))
                plt.setp(ax.get_xticklabels(), rotation=45, fontsize=9)
                ax.tick_params(axis='y', labelsize=9)

            plt.subplots_adjust(left=0.08, right=0.6, top=0.36, bottom=0.1, hspace=0.48)
        
            img_path = os.path.join(img_path_dir, f"{base_path}_group{i+1}_p{period_idx+1}.png")
            plt.savefig(img_path, dpi=200, bbox_inches='tight')
            plt.close(fig)      

            if save_path is not None:
                save_path.append((f"Group {i+1}", img_path))

    return save_path if save_path is not None else []


# ================== CREATE PDF REPORT ==================
def create_pdf_report(dfs, image_paths, comparison_images=None, filename="report.pdf"):
    if not isinstance(dfs, list):
        dfs = [dfs] if dfs is not None else []
    
    doc = SimpleDocTemplate(filename, pagesize=landscape(letter))
    elements = []
    styles = getSampleStyleSheet()

    # ================== TITLE ==================
    elements.append(Paragraph(f"Credit Risk Monitor Report - {datetime.now().strftime('%Y-%m-%d')}", styles['Title']))
    elements.append(Spacer(1, 15))

    
    # ================== PRICE TABLE ==================
    for df in dfs:
        if df is not None and not df.empty:        
            table_data = [df.columns.tolist()] + df.values.tolist()
            table = Table(table_data)
            table.setStyle(TableStyle([
                ('BACKGROUND', (0,0), (-1,0), colors.grey),
                ('TEXTCOLOR', (0,0), (-1,0), colors.whitesmoke),
                ('ALIGN', (0,0), (-1,-1), 'CENTER'),
                ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
                ('FONTSIZE', (0,0), (-1,-1), 8),
                ('BOTTOMPADDING', (0,0), (-1,0), 8),
                ('BACKGROUND', (0,1), (-1,-1), colors.beige),
                ('GRID', (0,0), (-1,-1), 1, colors.black)
            ]))
            elements.append(table)
            elements.append(Spacer(1, 15))

   # ================== LEGEND / EXPLANATION ==================
    elements.append(Paragraph("Ticker Explanations", styles['Heading2']))
    elements.append(Spacer(1, 8))

    legend_text = """
    DX-Y.NYB : US Dollar Index (DXY)<br/>
    ^VIX     : CBOE Volatility Index (Market Fear Gauge)<br/>
    ^IRX     : 13-Week Treasury Bill Yield<br/>
    ^FVX     : 5-Year Treasury Yield<br/>
    ^TNX     : 10-Year Treasury Yield<br/>
    ^TYX     : 30-Year Treasury Yield<br/>
    ^IXF     : Financial Sector Index (or similar)<br/>
    HYG      : iShares iBoxx High Yield Corporate Bond ETF<br/>
    XLF      : Financial Select Sector SPDR ETF<br/>
    KRE      : SPDR S&P Regional Banking ETF
    """

    legend_style = ParagraphStyle(
        'LegendStyle',
        parent=styles['Normal'],
        fontSize=9,
        leading=14,
        spaceAfter=20
    )

    elements.append(Paragraph(legend_text, legend_style))
    elements.append(PageBreak())

    # ================== INDIVIDUAL PLOTS ==================    
    # Create a centered title style
    centered_title = ParagraphStyle(
        'CenteredTitle',
        parent=styles['Heading2'],
        alignment=1,         
        fontSize=16,         
        spaceAfter=20
    )
    if image_paths:
        elements.append(Spacer(1, 120))
        elements.append(Paragraph("Individual Ticker Trends Over Years", centered_title))
        elements.append(Spacer(1, 20))
        for ticker, img_path in image_paths:
            if os.path.exists(img_path):
                elements.append(Image(img_path, width=750, height=450))
                elements.append(PageBreak())
            
    # ================== COMPARISON PLOTS ==================
    if comparison_images:
        elements.append(Spacer(1, 120))
        elements.append(Paragraph("Group Comparisons of Selected Tickers", centered_title))
        elements.append(Spacer(1, 20))
        for group, img_path in (comparison_images):
            if os.path.exists(img_path):
                elements.append(Image(img_path, width=750, height=450))
                elements.append(PageBreak())
                
    doc.build(elements)
    print(f"PDF report saved: {filename}")
    
    # Clean up temp images
    for _, img_path in image_paths:
        try:
            if os.path.exists(img_path):
                os.remove(img_path)
        except:
            pass      
    if comparison_images:
        for _, img_path in comparison_images:
            try:
                if os.path.exists(img_path):
                    os.remove(img_path)
            except:
                pass

                
# ================== MAIN ==================
if __name__ == "__main__":
    data_start_date = '2021-01-01'
    data_end_date = datetime.now().strftime('%Y-%m-%d')       
    tickers = ['DX-Y.NYB', '^VIX', '^IRX', '^FVX', '^TNX', '^TYX', '^IXF', 'HYG', 'XLF', 'KRE']
    comparison_groups = [['DX-Y.NYB', '^VIX'], ['HYG', '^TNX'], ['^IRX', '^FVX', '^TNX', '^TYX'], ['^IXF', 'XLF', 'KRE']]
    periods = [['2025-10-01', data_end_date], ['2024-04-01', data_end_date], ['2021-04-01', data_end_date] ] 
    
    print("Starting historical analysis...")
    df, df_v= download_historical_data(tickers, data_start_date, data_end_date)
   
    if df.empty:
        print("No data downloaded.")
        exit()   
 
    df_price = monitor_tickets(df, tickers)
    
    image_paths = []
    comparison_images = []
    
    plot_price_only(df, df_v, tickers, periods, base_path="ma", save_path=image_paths)
    plot_comparison(df, comparison_groups, periods, base_path="comparison", save_path=comparison_images)
   
    filename=f"CreditRisk_Monitor_{data_end_date.replace('-','')}.pdf"
    create_pdf_report(df_price, image_paths, comparison_images, filename=filename)


Starting historical analysis...
Download completed. Shape: (1334, 10)

=== Credit Risk Monitor - 2026-04-23 17:07 ===

  Ticker  Last Close   1W %     1M %     1Q %     1Y %     5Y %       MA9    MA20    MA50    MA200    52W Low-High
DX-Y.NYB       98.83     0.6%    -0.4%     0.5%    -5.4%    10.0%   98.31   99.06   98.78    98.52    96.2 - 110.0
    ^VIX       19.31     7.6%   -20.3%    30.0%    27.0%   -28.4%   18.63   22.01   22.48    18.24     12.8 - 52.3
    ^IRX        3.60    -0.4%    -0.1%     0.8%   -18.7%  5186.8%    3.60    3.60    3.60     3.77       3.5 - 4.4
    ^FVX        3.95     1.0%     4.5%     6.4%    -5.7%  1016.4%    3.90    3.94    3.82     3.76       3.5 - 4.6
    ^TNX        4.32     0.3%     2.7%     4.4%    -0.4%   371.4%    4.28    4.31    4.22     4.19       4.0 - 4.8
    ^TYX        4.92    -0.2%     1.3%     2.7%     8.3%   197.0%    4.90    4.90    4.83     4.80       4.3 - 5.1
    ^IXF     7195.79     0.1%     5.8%    -4.4%     8.9%    41.7% 7202.38 69